## Импорт библиотек и настройки окружения

In [3]:
import pandas as pd

from sklearn.ensemble import RandomForestRegressor

import mlflow
import mlflow.sklearn

RANDOM_SEED = 42

In [2]:
import warnings

warnings.filterwarnings("ignore")

In [4]:
from config import (
    MLFLOW_PORT
)
mlflow.set_tracking_uri("http://localhost:{}".format(MLFLOW_PORT))

Проверка подключения к MLflow

In [5]:
mlflow.search_experiments(max_results=3)

[<Experiment: artifact_location='mlflow-artifacts:/', creation_time=1780514331341, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1780514331341, lifecycle_stage='active', name='geolocation_popularity_random_forest', tags={}, trace_location=None, workspace='default'>,
 <Experiment: artifact_location='s3://mlflow-bucket/mlflow/0', creation_time=1780510990264, effective_trace_archival_retention=None, experiment_id='0', last_update_time=1780510990264, lifecycle_stage='active', name='Default', tags={}, trace_location=None, workspace='default'>]

## Подготовка датасета

In [6]:
class TargetEncoder:

    def __init__(
        self,
        smoothing: float = 0.0
    ) -> None:
        assert 0.0 <= smoothing <= 1.0, "smoothing must be in [0, 1]"
        self.smoothing = smoothing
        self.global_mean_: float = None
        self.encoding_maps_: dict[str, dict] = {}

    def fit(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        columns: list[str]
    ) -> "TargetEncoder":
        self.columns = columns
        self.global_mean_ = y.mean()
        for col in columns:
            cat_means = (
                pd.concat([X[col], y], axis=1)
                .groupby(col)[y.name]
                .mean()
            )
            encoded = (1 - self.smoothing) * cat_means + self.smoothing * self.global_mean_
            self.encoding_maps_[col] = encoded.to_dict()
        return self
    
    def transform(
        self,
        X: pd.DataFrame
    ) -> pd.DataFrame:
        X = X.copy()
        for col in self.columns:
            X[col] = X[col].map(self.encoding_maps_[col]).fillna(self.global_mean_)
        return X

    def fit_transform(
        self,
        X: pd.DataFrame,
        y: pd.Series,
        columns: list[str]
    ) -> pd.DataFrame:
        return self.fit(X, y, columns).transform(X)

In [7]:
df_train = pd.read_csv("./dataset/atm_train_split.csv", encoding='utf-8')
df_test = pd.read_csv("./dataset/atm_test_split.csv", encoding='utf-8')

In [8]:
df_train.reset_index(drop=True, inplace=True)

In [9]:
cat_cols = ["region", "atm_group"]

encoder = TargetEncoder(smoothing=0.1)
df_train_encoded = encoder.fit_transform(df_train, df_train["target"], columns=cat_cols)

df_test_encoded = encoder.transform(df_test)

In [10]:
FEATURE_COLS = [c for c in df_train.columns if c != "target"]

X_train = df_train[FEATURE_COLS].copy()
y_train = df_train["target"]

X_test = df_test[FEATURE_COLS].copy()
y_test = df_test["target"]

X_train_encoded = df_train_encoded[FEATURE_COLS]
X_test_encoded = df_test_encoded[FEATURE_COLS]

In [11]:
X_train["atm_group"] = X_train["atm_group"].astype(str)
X_test["atm_group"] = X_test["atm_group"].astype(str)

for col in cat_cols:
    X_train[col] = X_train[col].astype("category")
    X_test[col] = X_test[col].astype("category")

## Тестовое предсказание

In [15]:
registered_model_name = "geolocation_popularity_random_forest_tuned"
loaded_model = mlflow.pyfunc.load_model(f"models:/{registered_model_name}@prd")

sample_pred = loaded_model.predict(X_test_encoded)
sample_pred

array([-0.02898361, -0.01023222,  0.1621237 , ..., -0.05683536,
       -0.04913877,  0.02736693], shape=(1217,))